# SqlSum

[Codility task](https://app.codility.com/programmers/trainings/6/sql_sum/) - Exercise 6, SQL - *elementary*

Given

```sql
create table elements (
    v integer not null
);
```

write a query that returns the sum of the numbers in column `v`. For the values `2, 10, 20, 10` it must return `42`.

The SQL lives in `solution.sql` (the file you paste into Codility). This notebook loads `setup.sql` into an in-memory **SQLite** database with Python's `sqlite3`, runs the solution and checks it.

In [ ]:
import sqlite3
from pathlib import Path

HERE = Path.cwd()          # Jupyter starts the kernel in the notebook's folder


def fresh_db(with_example=True):
    """New in-memory SQLite database; optionally loaded with setup.sql (tables + example data)."""
    conn = sqlite3.connect(":memory:")
    if with_example:
        conn.executescript((HERE / "setup.sql").read_text(encoding="utf-8"))
    return conn


def run(conn, sql_file="solution.sql"):
    """Execute the single query stored in `sql_file` and return all its rows."""
    return conn.execute((HERE / sql_file).read_text(encoding="utf-8")).fetchall()


def show(rows, headers):
    """Tiny table printer (no pandas needed)."""
    table = [headers] + [[str(c) for c in row] for row in rows]
    widths = [max(len(r[i]) for r in table) for i in range(len(headers))]
    for n, row in enumerate(table):
        print(" | ".join(cell.ljust(w) for cell, w in zip(row, widths)))
        if n == 0:
            print("-+-".join("-" * w for w in widths))

## The solution

In [ ]:
print((HERE / "solution.sql").read_text(encoding="utf-8"))

`SUM(v)` is an aggregate: without `GROUP BY` the whole table collapses into a single row.

## Official example

In [ ]:
conn = fresh_db()
rows = run(conn)
show(rows, ["sum(v)"])
assert rows == [(42,)]
print("\nexample OK")

## Edge cases

- empty table: `SUM` of zero rows is `NULL` (not `0`);
- a single row and negative numbers;
- values whose sum does not fit in 32 bits (SQLite uses 64-bit integers; PostgreSQL's `SUM(integer)` returns `bigint`).

In [ ]:
def sum_of(values):
    conn = fresh_db()
    conn.execute("delete from elements")          # keep the table, drop the example rows
    conn.executemany("insert into elements (v) values (?)", [(v,) for v in values])
    return run(conn)


assert sum_of([]) == [(None,)]                            # NULL for an empty table
assert sum_of([7]) == [(7,)]
assert sum_of([-5, 5]) == [(0,)]
assert sum_of([-3, -4]) == [(-7,)]
assert sum_of([2**31 - 1] * 3) == [(3 * (2**31 - 1),)]    # beyond 32 bits
print("edge cases OK")

Whether an empty table should give `NULL` or `0` is a design choice. If you need `0`, change the query to `select coalesce(sum(v), 0) from elements;`.

## Randomised check against Python's `sum`

In [ ]:
import random

rng = random.Random(42)
for _ in range(500):
    values = [rng.randint(-1000, 1000) for _ in range(rng.randint(1, 50))]
    assert sum_of(values) == [(sum(values),)], values
print("SQL result equals sum() on 500 random tables")

## Notes

- Related aggregates: `COUNT`, `AVG`, `MIN`, `MAX`. All ignore `NULL` values, but the column here is `NOT NULL`.
- Aggregate functions return exactly one row when there is no `GROUP BY`, even on an empty table.
- The same query runs unchanged on PostgreSQL, the other dialect offered by Codility.